In [1]:

import os
import parse_labels
import importlib

importlib.reload(parse_labels)

# gets the labels
base_path = os.getcwd()
input_path = "raw_data/manifest-xxn3N2Qq630907925598003437/TCGA-KIRC"
all_dirs = os.listdir(input_path)


label_path = os.path.join(base_path, "raw_data", "labels.txt")
data_path = os.path.join(base_path, "raw_data", "manifest-xxn3N2Qq630907925598003437/TCGA-KIRC")

map = parse_labels.get_all_dcm_files(label_path, data_path)


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Subset
import model
import importlib

importlib.reload(model)

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [ ]:
importlib.reload(model)
import numpy as np

# Get patient scan series data (grouped by patient AND scan series)
print("Fetching patient scan series data...")
series_data = parse_labels.get_all_dcm_files(label_path, data_path)
print(f"Found {len(series_data)} scan series total\n")



Fetching patient scan series data...
Found 2652 scan series total

Example scan series:
  Patient ID: TCGA-B0-4698
  Series ID: /home/petedowney/github/KidneyCancerPrediction/raw_data/manifest-xxn3N2Qq630907925598003437/TCGA-KIRC/TCGA-B0-4698/11-22-1985-NA-CT ABDOMEN-45292/2.000000-C-14369
  Grade: 4
  Number of DICOM files listed: 56

  First 3 files:
    1-01.dcm
    1-02.dcm
    1-03.dcm


In [9]:
# Preload scan series volumes into memory
print("Preloading scan series volumes...")
preloaded_data = model.preload_dcm_files(
    series_data, 
    target_shape=(64, 64, 64),
    max_files=None   # Start with small subset to test
)
print(f"Successfully preloaded {len(preloaded_data)} scan series volumes")

Preloading scan series volumes...
Loading all 2652 scan series


Preloading Scan Series: 100%|██████████| 2652/2652 [09:10<00:00,  4.82series/s]

Successfully preloaded 2652 scan series volumes


In [10]:
# Create 80/20 train/validation split
train_size = int(0.8 * len(preloaded_data))
indices = list(range(len(preloaded_data)))
np.random.seed(42)
np.random.shuffle(indices)

train_indices = indices[:train_size]
val_indices = indices[train_size:]

print(f"Train scan series: {len(train_indices)}")
print(f"Val scan series: {len(val_indices)}\n")

# Create dataloaders
train_dataset = model.DCMDataset(series_data, target_shape=(64, 64, 64), preloaded_data=preloaded_data)
val_dataset = model.DCMDataset(series_data, target_shape=(64, 64, 64), preloaded_data=preloaded_data)

train_subset = Subset(train_dataset, train_indices)
val_subset = Subset(val_dataset, val_indices)

train_loader = torch.utils.data.DataLoader(
    train_subset, batch_size=2, shuffle=True, num_workers=0
)
val_loader = torch.utils.data.DataLoader(
    val_subset, batch_size=2, shuffle=False, num_workers=0
)

# Check dataset sizes
print(f"Train subset size: {len(train_subset)}")
print(f"Val subset size: {len(val_subset)}")
print(f"Train batches per epoch: {len(train_loader)}")
print(f"Val batches per epoch: {len(val_loader)}\n")

# Initialize model
net = model.SimpleCNN(num_classes=4).to(device)

Train scan series: 2121
Val scan series: 531

Train subset size: 2121
Val subset size: 531
Train batches per epoch: 1061
Val batches per epoch: 266



In [11]:
importlib.reload(model)
# Train the model
history = model.train_model(
    net, 
    train_loader, 
    val_loader, 
    device, 
    num_epochs=10, 
    learning_rate=0.001
)

Epoch 1/10 Val: 100%|██████████| 266/266 [00:03<00:00, 72.17batch/s]


Train Loss: 4.5813 | Train Acc: 39.89% | Val Loss: 1.1916 | Val Acc: 39.92%


Epoch 2/10 Val: 100%|██████████| 266/266 [00:02<00:00, 96.02batch/s]


Train Loss: 1.1329 | Train Acc: 41.07% | Val Loss: 1.1027 | Val Acc: 42.75%


Epoch 3/10 Val: 100%|██████████| 266/266 [00:02<00:00, 95.56batch/s]


...

Train Loss: 1.0555 | Train Acc: 42.86% | Val Loss: 1.0573 | Val Acc: 42.94%


Epoch 9/10 Val: 100%|██████████| 266/266 [00:03<00:00, 85.08batch/s]


Train Loss: 1.0555 | Train Acc: 41.35% | Val Loss: 1.0562 | Val Acc: 42.94%


Epoch 10/10 Val: 100%|██████████| 266/266 [00:03<00:00, 81.54batch/s]

Train Loss: 1.0563 | Train Acc: 42.86% | Val Loss: 1.0559 | Val Acc: 42.94%
Training complete!


In [12]:
# Save the trained model
model_path = os.path.join(base_path, "kidney_cancer_model.pth")
torch.save(net.state_dict(), model_path)
print(f"Model saved to {model_path}")

Model saved to /home/petedowney/github/KidneyCancerPrediction/kidney_cancer_model.pth
